<a href="https://colab.research.google.com/github/AabhinavAnand/Netflix-Data-Visualization-/blob/main/Injury_Risk_Prediction_MultiSensor_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Injury Risk Prediction using Multi-Sensor Fusion
### Deep Learning-Based Early Warning System

---

**Abstract:**  
This project presents a multi-sensor fusion framework for predicting high-risk injury phases during physical activity. We fuse data from accelerometers, gyroscopes, EMG sensors, and heart rate monitors using a hybrid CNN-LSTM-Attention architecture. The model achieves early warning of injury risk 2–5 seconds before critical movement phases.

**Sensors Used:**
- 📡 **Accelerometer** (3-axis): Linear acceleration & impact forces
- 🔄 **Gyroscope** (3-axis): Angular velocity & rotation
- 💪 **EMG** (2-channel): Muscle activation patterns
- ❤️ **Heart Rate Monitor**: Physiological fatigue indicator
- 🦴 **Force Plate**: Ground reaction forces (simulated)

**Pipeline:**  
`Raw Sensor Data → Preprocessing → Feature Extraction → Sensor Fusion → CNN-LSTM-Attention → Risk Prediction → Early Warning`


## 📦 Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib seaborn scipy tqdm
!pip install -q shap captum  # For explainability
print('✅ All dependencies installed!')

## 📚 Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import signal
from scipy.stats import kurtosis, skew
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              f1_score, accuracy_score)
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import random
import os
import json
from collections import defaultdict

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
print(f'🔥 PyTorch Version: {torch.__version__}')

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Libraries loaded!')

## 🔬 Step 3: Synthetic Multi-Sensor Data Generation
> Simulates realistic biomechanical data from wearable sensors during athletic activities

In [ ]:
class MultiSensorDataGenerator:
    """
    Generates realistic multi-sensor biomechanical data.
    Simulates 5 sensor modalities across 3 risk categories.
    """

    def __init__(self, fs=100, window_size=200, n_subjects=50):
        """
        fs: Sampling frequency (Hz)
        window_size: Samples per window (2 seconds at 100Hz)
        n_subjects: Number of simulated athletes
        """
        self.fs = fs
        self.window_size = window_size
        self.n_subjects = n_subjects
        self.t = np.linspace(0, window_size/fs, window_size)

        # Risk labels: 0=Low, 1=Medium, 2=High
        self.RISK_LABELS = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
        self.RISK_COLORS = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}

    def _add_noise(self, signal_data, noise_level=0.05):
        """Add realistic sensor noise"""
        return signal_data + np.random.normal(0, noise_level * np.std(signal_data), len(signal_data))

    def _generate_accelerometer(self, risk_level, subject_var=1.0):
        """3-axis accelerometer (g) - captures impact forces"""
        t = self.t
        base_freq = 2.0 + risk_level * 0.5

        # X-axis: lateral movement
        ax = subject_var * (np.sin(2*np.pi*base_freq*t) +
                            0.3*np.sin(2*np.pi*3*base_freq*t))
        # Y-axis: anterior-posterior
        ay = subject_var * (np.cos(2*np.pi*base_freq*t) +
                            0.2*np.sin(2*np.pi*2*base_freq*t))
        # Z-axis: vertical (impact)
        az = 1.0 + subject_var * (1.5 + risk_level * 0.8) * np.abs(np.sin(2*np.pi*base_freq*t))

        if risk_level == 2:  # High risk: sudden spikes
            spike_idx = np.random.choice(len(t), size=3, replace=False)
            az[spike_idx] += np.random.uniform(2, 4, 3)
            ax[spike_idx] += np.random.uniform(1, 2, 3)

        return np.stack([
            self._add_noise(ax),
            self._add_noise(ay),
            self._add_noise(az)
        ], axis=1)

    def _generate_gyroscope(self, risk_level, subject_var=1.0):
        """3-axis gyroscope (°/s) - angular velocity"""
        t = self.t
        base_freq = 1.5 + risk_level * 0.3

        gx = subject_var * (45 + risk_level * 20) * np.sin(2*np.pi*base_freq*t)
        gy = subject_var * (30 + risk_level * 15) * np.cos(2*np.pi*base_freq*t)
        gz = subject_var * (20 + risk_level * 25) * np.sin(2*np.pi*2*base_freq*t)

        if risk_level >= 1:  # Abnormal rotation pattern
            gz += subject_var * risk_level * 15 * np.sin(2*np.pi*4*base_freq*t)

        return np.stack([
            self._add_noise(gx, 0.03),
            self._add_noise(gy, 0.03),
            self._add_noise(gz, 0.03)
        ], axis=1)

    def _generate_emg(self, risk_level, subject_var=1.0):
        """2-channel EMG (mV) - muscle activation"""
        t = self.t
        # Channel 1: Quadriceps
        quad_base = subject_var * (0.5 + risk_level * 0.3)
        emg1 = quad_base * np.abs(np.random.randn(len(t)))

        # Fatigue signature in high risk
        if risk_level == 2:
            # Decrease in amplitude with frequency shifts (fatigue)
            fatigue_envelope = np.linspace(1.0, 0.6, len(t))
            emg1 *= fatigue_envelope
            emg1 += 0.1 * np.sin(2*np.pi*60*t)  # EMG noise artifact

        # Channel 2: Hamstring (co-contraction in risk)
        ham_base = subject_var * (0.3 + risk_level * 0.4)
        emg2 = ham_base * np.abs(np.random.randn(len(t)))

        # Apply bandpass filter (20-450 Hz typical for EMG)
        b, a = signal.butter(4, [20/(self.fs/2), min(0.99, 450/(self.fs/2))], btype='band')
        emg1 = signal.filtfilt(b, a, emg1)
        emg2 = signal.filtfilt(b, a, emg2)

        return np.stack([
            self._add_noise(emg1, 0.02),
            self._add_noise(emg2, 0.02)
        ], axis=1)

    def _generate_heart_rate(self, risk_level, subject_var=1.0):
        """Heart rate (BPM) - physiological fatigue indicator"""
        t = self.t
        # Baseline HR increases with risk (fatigue)
        base_hr = 80 + risk_level * 30 + subject_var * 10
        # HRV decreases with risk
        hrv = (10 - risk_level * 3) * np.sin(2*np.pi*0.25*t)
        hr = base_hr + hrv + np.random.normal(0, 2, len(t))
        return hr.reshape(-1, 1)

    def _generate_force_plate(self, risk_level, subject_var=1.0):
        """Ground Reaction Force (N/kg) - asymmetry detection"""
        t = self.t
        body_weight = subject_var * 70  # kg
        g = 9.81

        # Normal walking/running GRF pattern
        grf_base = body_weight * g * (1.2 + risk_level * 0.4)

        # Double-peak GRF pattern
        step_freq = 1.8 + risk_level * 0.3
        grf = grf_base * (np.abs(np.sin(2*np.pi*step_freq*t)) +
                           0.3 * np.abs(np.sin(2*np.pi*2*step_freq*t)))

        if risk_level == 2:  # Asymmetric loading
            grf += grf_base * 0.3 * np.sin(2*np.pi*0.5*t)

        return self._add_noise(grf, 0.02).reshape(-1, 1)

    def generate_sample(self, risk_level):
        """Generate one multi-sensor sample for a given risk level"""
        subject_var = np.random.uniform(0.8, 1.2)  # Inter-subject variability

        acc = self._generate_accelerometer(risk_level, subject_var)  # (T, 3)
        gyro = self._generate_gyroscope(risk_level, subject_var)     # (T, 3)
        emg = self._generate_emg(risk_level, subject_var)            # (T, 2)
        hr = self._generate_heart_rate(risk_level, subject_var)      # (T, 1)
        grf = self._generate_force_plate(risk_level, subject_var)    # (T, 1)

        # Concatenate all sensors: (T, 10)
        return np.concatenate([acc, gyro, emg, hr, grf], axis=1)

    def generate_dataset(self, samples_per_class=500):
        """Generate complete dataset with class balance"""
        print('🔄 Generating multi-sensor dataset...')
        X, y = [], []

        for risk_level in range(3):
            print(f'  📊 Generating {samples_per_class} samples for {self.RISK_LABELS[risk_level]}...')
            for _ in range(samples_per_class):
                sample = self.generate_sample(risk_level)
                X.append(sample)
                y.append(risk_level)

        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.int64)

        print(f'✅ Dataset shape: X={X.shape}, y={y.shape}')
        print(f'   Sensor channels: Acc(3) + Gyro(3) + EMG(2) + HR(1) + GRF(1) = 10')
        return X, y

# Generate data
gen = MultiSensorDataGenerator(fs=100, window_size=200)
X, y = gen.generate_dataset(samples_per_class=500)
print(f'\n📐 Final Dataset: {X.shape[0]} samples × {X.shape[1]} timesteps × {X.shape[2]} channels')

## 📊 Step 4: Exploratory Data Analysis

In [ ]:
CHANNEL_NAMES = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z',
                 'EMG_Quad', 'EMG_Ham', 'HR', 'GRF']
RISK_LABELS = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
RISK_COLORS = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}

fig, axes = plt.subplots(5, 2, figsize=(18, 20))
fig.suptitle('Multi-Sensor Signal Visualization by Risk Level', fontsize=16, fontweight='bold', y=0.98)

t = np.linspace(0, 2, 200)
sensor_groups = [
    (0, 'Accelerometer X (g)'),
    (2, 'Accelerometer Z (g)'),
    (3, 'Gyroscope X (°/s)'),
    (5, 'Gyroscope Z (°/s)'),
    (6, 'EMG Quadriceps (mV)'),
    (7, 'EMG Hamstring (mV)'),
    (8, 'Heart Rate (BPM)'),
    (9, 'Ground Reaction Force (N)'),
]

for idx, (ch, name) in enumerate(sensor_groups):
    ax = axes[idx//2][idx%2]
    for risk in range(3):
        mask = y == risk
        sample = X[mask][0]  # First sample
        ax.plot(t, sample[:, ch], color=RISK_COLORS[risk],
                label=RISK_LABELS[risk], alpha=0.85, linewidth=1.5)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# 9th plot: class distribution
ax = axes[4][0]
counts = [np.sum(y==i) for i in range(3)]
bars = ax.bar([RISK_LABELS[i] for i in range(3)], counts,
              color=[RISK_COLORS[i] for i in range(3)], edgecolor='black', linewidth=0.8)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(count), ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Sample Count')

# 10th plot: sensor correlation heatmap
ax = axes[4][1]
sample_df = pd.DataFrame(X[y==2][0], columns=CHANNEL_NAMES)
corr = sample_df.corr()
sns.heatmap(corr, ax=ax, cmap='RdYlGn', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Sensor Correlation (High Risk)', fontweight='bold')

plt.tight_layout()
plt.savefig('sensor_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots generated!')

## ⚙️ Step 5: Feature Engineering

In [ ]:
class FeatureExtractor:
    """
    Extracts time-domain, frequency-domain, and cross-sensor features.
    Used for baseline ML models and feature importance analysis.
    """

    def extract_time_domain(self, x):
        """x: (T, C)"""
        feats = []
        feats.extend(np.mean(x, axis=0))        # Mean
        feats.extend(np.std(x, axis=0))         # Std
        feats.extend(np.max(x, axis=0))         # Max
        feats.extend(np.min(x, axis=0))         # Min
        feats.extend(np.max(x,axis=0)-np.min(x,axis=0)) # Range
        feats.extend(kurtosis(x, axis=0))       # Kurtosis (peakedness)
        feats.extend(skew(x, axis=0))           # Skewness
        feats.extend(np.sqrt(np.mean(x**2, axis=0)))  # RMS
        feats.extend(np.percentile(x, 25, axis=0))   # Q1
        feats.extend(np.percentile(x, 75, axis=0))   # Q3
        return np.array(feats)

    def extract_freq_domain(self, x, fs=100):
        """FFT-based features"""
        feats = []
        for c in range(x.shape[1]):
            fft_vals = np.abs(np.fft.rfft(x[:, c]))
            freqs = np.fft.rfftfreq(len(x[:, c]), d=1/fs)
            # Dominant frequency
            feats.append(freqs[np.argmax(fft_vals)])
            # Spectral energy in bands
            low = np.sum(fft_vals[(freqs >= 0.5) & (freqs < 3)])
            mid = np.sum(fft_vals[(freqs >= 3) & (freqs < 10)])
            high = np.sum(fft_vals[(freqs >= 10) & (freqs < 45)])
            feats.extend([low, mid, high])
            # Spectral entropy
            psd = fft_vals**2
            psd_norm = psd / (np.sum(psd) + 1e-10)
            entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-10))
            feats.append(entropy)
        return np.array(feats)

    def extract_cross_sensor(self, x):
        """Cross-sensor correlation features (sensor fusion at feature level)"""
        # Correlation between acc and gyro
        acc = x[:, :3]
        gyro = x[:, 3:6]
        emg = x[:, 6:8]
        hr = x[:, 8]
        grf = x[:, 9]

        feats = []
        # Acc magnitude
        acc_mag = np.sqrt(np.sum(acc**2, axis=1))
        feats.extend([np.mean(acc_mag), np.std(acc_mag), np.max(acc_mag)])
        # Gyro magnitude
        gyro_mag = np.sqrt(np.sum(gyro**2, axis=1))
        feats.extend([np.mean(gyro_mag), np.std(gyro_mag)])
        # EMG Co-contraction Index
        cci = np.mean(np.minimum(np.abs(emg[:, 0]), np.abs(emg[:, 1])))
        feats.append(cci)
        # HR-GRF correlation (loading during fatigue)
        hr_grf_corr = np.corrcoef(hr, grf)[0, 1] if np.std(hr) > 0 and np.std(grf) > 0 else 0
        feats.append(hr_grf_corr)
        # Jerk (rate of change of acceleration)
        jerk = np.diff(acc_mag)
        feats.extend([np.mean(np.abs(jerk)), np.max(np.abs(jerk))])
        return np.array(feats)

    def extract_all(self, x):
        td = self.extract_time_domain(x)
        fd = self.extract_freq_domain(x)
        cs = self.extract_cross_sensor(x)
        return np.concatenate([td, fd, cs])

# Extract features
print('⚙️  Extracting features...')
fe = FeatureExtractor()
X_feat = np.array([fe.extract_all(X[i]) for i in tqdm(range(len(X)))])
print(f'✅ Feature matrix: {X_feat.shape} ({X_feat.shape[1]} features per sample)')

## 🔄 Step 6: Dataset & DataLoader

In [ ]:
class SensorFusionDataset(Dataset):
    """PyTorch Dataset for multi-sensor time series"""

    def __init__(self, X, y, scaler=None, fit_scaler=False):
        # Normalize per-channel across time axis
        T, C = X.shape[1], X.shape[2]
        X_flat = X.reshape(-1, C)

        if scaler is None:
            scaler = StandardScaler()
        if fit_scaler:
            scaler.fit(X_flat)

        X_norm = scaler.transform(X_flat).reshape(X.shape)
        self.X = torch.FloatTensor(X_norm)
        self.y = torch.LongTensor(y)
        self.scaler = scaler

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Train/Val/Test Split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

train_ds = SensorFusionDataset(X_train, y_train, fit_scaler=True)
val_ds = SensorFusionDataset(X_val, y_val, scaler=train_ds.scaler)
test_ds = SensorFusionDataset(X_test, y_test, scaler=train_ds.scaler)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'📦 Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'📦 Batches per epoch: {len(train_loader)}')

## 🧠 Step 7: CNN-LSTM-Attention Model Architecture
> Multi-scale temporal fusion for multi-sensor injury risk classification

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention for sensor importance weighting"""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        # x: (B, T, C)
        w = self.fc(x.mean(dim=1))  # Global avg pooling
        return x * w.unsqueeze(1)


class TemporalAttention(nn.Module):
    """Temporal self-attention to focus on high-risk movement phases"""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        # lstm_out: (B, T, H)
        scores = self.attn(lstm_out)                   # (B, T, 1)
        weights = torch.softmax(scores, dim=1)         # (B, T, 1)
        context = (lstm_out * weights).sum(dim=1)      # (B, H)
        return context, weights.squeeze(-1)             # context, attn_weights


class SensorFusionBlock(nn.Module):
    """Fuses multiple sensor streams with cross-attention"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Separate CNN branches per sensor group
        self.acc_branch = nn.Conv1d(3, out_channels//4, kernel_size=5, padding=2)
        self.gyro_branch = nn.Conv1d(3, out_channels//4, kernel_size=5, padding=2)
        self.emg_branch = nn.Conv1d(2, out_channels//4, kernel_size=3, padding=1)
        self.bio_branch = nn.Conv1d(2, out_channels//4, kernel_size=7, padding=3)
        self.fusion_bn = nn.BatchNorm1d(out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        # x: (B, T, 10) -> permute to (B, 10, T)
        x = x.permute(0, 2, 1)
        acc_out = self.act(self.acc_branch(x[:, :3, :]))   # (B, C/4, T)
        gyro_out = self.act(self.gyro_branch(x[:, 3:6, :]))
        emg_out = self.act(self.emg_branch(x[:, 6:8, :]))
        bio_out = self.act(self.bio_branch(x[:, 8:10, :]))
        fused = torch.cat([acc_out, gyro_out, emg_out, bio_out], dim=1)  # (B, C, T)
        fused = self.act(self.fusion_bn(fused))
        return fused.permute(0, 2, 1)  # (B, T, C)


class InjuryRiskNet(nn.Module):
    """
    Multi-Sensor Fusion CNN-LSTM-Attention Network

    Architecture:
    Input (B, T, 10) ->
    ChannelAttention -> SensorFusionBlock (parallel CNNs) ->
    Multi-scale CNN -> BiLSTM -> TemporalAttention -> MLP -> Output (3 classes)
    """

    def __init__(self, input_channels=10, hidden_dim=128, n_classes=3, dropout=0.3):
        super().__init__()
        self.n_classes = n_classes

        # Stage 1: Channel Attention (sensor weighting)
        self.channel_attn = ChannelAttention(input_channels)

        # Stage 2: Sensor Fusion Block (parallel CNN per sensor group)
        self.fusion_block = SensorFusionBlock(input_channels, hidden_dim)

        # Stage 3: Multi-scale CNN (extract temporal patterns at different scales)
        self.cnn_scales = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden_dim, hidden_dim//2, kernel_size=k, padding=k//2),
                nn.BatchNorm1d(hidden_dim//2),
                nn.GELU(),
                nn.Dropout(dropout/2)
            ) for k in [3, 7, 15]  # Short, medium, long-range patterns
        ])
        self.cnn_proj = nn.Linear((hidden_dim//2) * 3, hidden_dim)
        self.cnn_bn = nn.BatchNorm1d(hidden_dim)

        # Stage 4: Bidirectional LSTM (temporal dependencies)
        self.bilstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # Stage 5: Temporal Attention (focus on high-risk moments)
        self.temporal_attn = TemporalAttention(hidden_dim * 2)

        # Stage 6: Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, n_classes)
        )

    def forward(self, x, return_attention=False):
        # x: (B, T, C)

        # Channel attention - learn which sensors are most informative
        x = self.channel_attn(x)

        # Sensor fusion - parallel CNN per sensor group
        x = self.fusion_block(x)  # (B, T, H)

        # Multi-scale CNN
        x_perm = x.permute(0, 2, 1)  # (B, H, T)
        scale_outs = [conv(x_perm).permute(0, 2, 1) for conv in self.cnn_scales]  # each (B, T, H//2)
        x = torch.cat(scale_outs, dim=-1)  # (B, T, 3*H//2)
        x = F.gelu(self.cnn_bn(
            self.cnn_proj(x).permute(0, 2, 1)
        ).permute(0, 2, 1))  # (B, T, H)

        # BiLSTM
        lstm_out, _ = self.bilstm(x)  # (B, T, 2H)

        # Temporal attention
        context, attn_weights = self.temporal_attn(lstm_out)  # (B, 2H)

        # Classify
        logits = self.classifier(context)  # (B, 3)

        if return_attention:
            return logits, attn_weights
        return logits


# Instantiate model
model = InjuryRiskNet(input_channels=10, hidden_dim=128, n_classes=3, dropout=0.3).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'🧠 Model Architecture: InjuryRiskNet (CNN + BiLSTM + Attention)')
print(f'   Total Parameters: {total_params:,}')
print(f'   Trainable Parameters: {trainable_params:,}')
print(f'   Input: (Batch, 200 timesteps, 10 sensor channels)')
print(f'   Output: (Batch, 3 risk classes)')

## 🏋️ Step 8: Training with Advanced Techniques

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance in injury prediction"""
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


def train_epoch(model, loader, optimizer, criterion, scaler_amp):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            probs = F.softmax(logits, dim=1)
            total_loss += loss.item() * len(y_batch)
            preds = logits.argmax(1)
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return total_loss/total, correct/total, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# Training setup
EPOCHS = 60
LR = 1e-3

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = FocalLoss(gamma=2.0)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
amp_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

history = defaultdict(list)
best_val_acc = 0
best_model_state = None
patience_counter = 0
PATIENCE = 15

print(f'🏋️ Training for {EPOCHS} epochs on {DEVICE}...')
print('='*70)

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, amp_scaler)
    vl_loss, vl_acc, _, _, _ = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    # Save best model
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        saved = '💾'
    else:
        patience_counter += 1
        saved = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'Loss: {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc: {tr_acc:.4f}/{vl_acc:.4f} | '
              f'LR: {optimizer.param_groups[0]["lr"]:.6f} {saved}')

    if patience_counter >= PATIENCE:
        print(f'⏹️  Early stopping at epoch {epoch}')
        break

print(f'\n✅ Best Validation Accuracy: {best_val_acc:.4f}')

# Load best model
model.load_state_dict({k: v.to(DEVICE) for k, v in best_model_state.items()})
torch.save(model.state_dict(), 'best_injury_risk_model.pt')
print('💾 Best model saved!')

## 📈 Step 9: Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History - InjuryRiskNet', fontsize=14, fontweight='bold')

ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(ep, history['val_loss'], label='Val Loss', color='#e74c3c', linewidth=2)
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Focal Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, [a*100 for a in history['train_acc']], label='Train Acc', color='#2ecc71', linewidth=2)
axes[1].plot(ep, [a*100 for a in history['val_acc']], label='Val Acc', color='#f39c12', linewidth=2)
axes[1].axhline(best_val_acc*100, color='red', linestyle='--', alpha=0.5, label=f'Best Val: {best_val_acc*100:.1f}%')
axes[1].set_title('Accuracy Curves', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(ep, history['lr'], color='#9b59b6', linewidth=2)
axes[2].set_title('Learning Rate Schedule (Cosine Annealing)', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Step 10: Evaluation on Test Set

In [ ]:
# Test evaluation
test_loss, test_acc, test_preds, test_labels, test_probs = eval_epoch(model, test_loader, criterion)
f1 = f1_score(test_labels, test_preds, average='weighted')
auc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='weighted')

print('='*60)
print('📊 TEST RESULTS')
print('='*60)
print(f'  Accuracy:       {test_acc*100:.2f}%')
print(f'  Weighted F1:    {f1:.4f}')
print(f'  ROC-AUC (OvR):  {auc:.4f}')
print('='*60)
print('\n📋 Classification Report:')
print(classification_report(test_labels, test_preds,
                            target_names=['Low Risk', 'Med Risk', 'High Risk']))

In [ ]:
# Visualization: Confusion Matrix + ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Evaluation Metrics', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, ax=axes[0], annot=cm, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'],
            linewidths=0.5, cbar_kws={'label': 'Normalized'})
axes[0].set_title(f'Confusion Matrix (Acc={test_acc*100:.1f}%)', fontweight='bold')
axes[0].set_xlabel('Predicted Risk Level')
axes[0].set_ylabel('True Risk Level')

# ROC Curves (One-vs-Rest)
class_names = ['Low Risk', 'Medium Risk', 'High Risk']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(test_labels, classes=[0, 1, 2])

for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], test_probs[:, i])
    auc_i = roc_auc_score(y_bin[:, i], test_probs[:, i])
    axes[1].plot(fpr, tpr, color=color, linewidth=2.5,
                 label=f'{name} (AUC={auc_i:.3f})')

axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves (One-vs-Rest)', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚠️ Step 11: Early Warning System & Attention Visualization

In [ ]:
class EarlyWarningSystem:
    """
    Real-time injury risk early warning system.
    Monitors sliding windows and triggers alerts before high-risk phases.
    """

    def __init__(self, model, scaler, fs=100, window_size=200, step_size=20,
                 warning_threshold=0.65, critical_threshold=0.85,
                 warning_lead_time=3.0):
        self.model = model
        self.scaler = scaler
        self.fs = fs
        self.window_size = window_size
        self.step_size = step_size
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold
        self.warning_lead_time = warning_lead_time
        self.model.eval()

    def predict_window(self, window):
        """Predict risk for a single window of sensor data"""
        x = self.scaler.transform(window).reshape(1, *window.shape)
        x_t = torch.FloatTensor(x).to(DEVICE)
        with torch.no_grad():
            logits, attn = self.model(x_t, return_attention=True)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        return probs, attn.cpu().numpy()[0]

    def simulate_session(self, sensor_stream, n_windows=80):
        """Simulate real-time monitoring of an athletic session"""
        results = []
        total_samples = n_windows * self.step_size + self.window_size

        for i in range(n_windows):
            start = i * self.step_size
            end = start + self.window_size
            if end > len(sensor_stream):
                break

            window = sensor_stream[start:end]
            probs, attn = self.predict_window(window)

            risk_score = probs[2] * 1.0 + probs[1] * 0.5  # Weighted risk
            time_s = (start + self.window_size//2) / self.fs

            alert = 'NORMAL'
            if probs[2] > self.critical_threshold:
                alert = '🚨 CRITICAL'
            elif risk_score > self.warning_threshold:
                alert = '⚠️  WARNING'
            elif probs[1] > 0.5:
                alert = '🟡 CAUTION'

            results.append({
                'time': time_s,
                'probs': probs,
                'risk_score': risk_score,
                'predicted_class': np.argmax(probs),
                'alert': alert,
                'attention': attn
            })

        return results


# Generate a simulated athletic session (5 minutes)
print('🏃 Simulating athletic session...')
gen2 = MultiSensorDataGenerator(fs=100)

# Create a session with escalating risk
session_parts = [
    (gen2.generate_sample(0), 2000),   # Warm-up: Low risk
    (gen2.generate_sample(1), 2000),   # Mid session: Medium risk
    (gen2.generate_sample(2), 2000),   # Late session: High risk (fatigue)
    (gen2.generate_sample(1), 1000),   # Recovery
]

session_stream = []
for sample, n in session_parts:
    # Tile the sample to get required length
    repeats = int(np.ceil(n / len(sample)))
    tiled = np.tile(sample, (repeats, 1))[:n]
    # Add drift noise
    tiled += np.random.normal(0, 0.02, tiled.shape)
    session_stream.append(tiled)

session_stream = np.concatenate(session_stream, axis=0).astype(np.float32)
print(f'📡 Session length: {len(session_stream)/100:.1f} seconds ({len(session_stream)} samples)')

# Run early warning system
ews = EarlyWarningSystem(model, train_ds.scaler)
results = ews.simulate_session(session_stream, n_windows=100)

# Print some key alerts
print('\n🔔 Early Warning Events:')
print(f'{"Time":>8} | {"Risk Score":>10} | {"Alert"}')
print('-'*45)
for r in results[::8]:
    print(f'{r["time"]:>7.1f}s | {r["risk_score"]:>10.3f} | {r["alert"]}')

In [ ]:
# Visualize Early Warning Timeline
times = [r['time'] for r in results]
risk_scores = [r['risk_score'] for r in results]
high_probs = [r['probs'][2] for r in results]
med_probs = [r['probs'][1] for r in results]
low_probs = [r['probs'][0] for r in results]
pred_classes = [r['predicted_class'] for r in results]

fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Real-Time Injury Risk Early Warning System', fontsize=16, fontweight='bold')

# Panel 1: Risk Score Timeline
ax = axes[0]
ax.fill_between(times, risk_scores, alpha=0.3, color='#e74c3c')
ax.plot(times, risk_scores, color='#c0392b', linewidth=2, label='Risk Score')
ax.axhline(ews.warning_threshold, color='#f39c12', linestyle='--', linewidth=2, label=f'Warning ({ews.warning_threshold})')
ax.axhline(ews.critical_threshold, color='#e74c3c', linestyle='--', linewidth=2, label=f'Critical ({ews.critical_threshold})')

# Add phase backgrounds
ax.axvspan(0, 20, alpha=0.1, color='green', label='Low Risk Phase')
ax.axvspan(20, 40, alpha=0.1, color='orange', label='Medium Risk Phase')
ax.axvspan(40, 60, alpha=0.1, color='red', label='High Risk Phase')

ax.set_ylabel('Risk Score', fontsize=12)
ax.set_title('Composite Risk Score Over Time', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)

# Panel 2: Class Probabilities (stacked area)
ax = axes[1]
ax.stackplot(times, low_probs, med_probs, high_probs,
             labels=['Low Risk', 'Medium Risk', 'High Risk'],
             colors=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.8)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Risk Class Probabilities (Stacked)', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Panel 3: Predicted Class with Early Warning markers
ax = axes[2]
scatter_colors = [['#2ecc71', '#f39c12', '#e74c3c'][c] for c in pred_classes]
ax.scatter(times, pred_classes, c=scatter_colors, s=40, zorder=3, edgecolors='black', linewidth=0.5)
ax.plot(times, pred_classes, color='gray', linewidth=0.5, alpha=0.5)

# Mark critical events
for r in results:
    if r['alert'] == '🚨 CRITICAL':
        ax.axvline(r['time'], color='red', alpha=0.3, linewidth=1)
    elif '⚠️' in r['alert']:
        ax.axvline(r['time'], color='orange', alpha=0.2, linewidth=1)

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Low', 'Medium', 'High'])
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Predicted Risk', fontsize=12)
ax.set_title('Predicted Risk Class + Warning Events (red=critical, orange=warning)', fontweight='bold')
ax.grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlim(min(times), max(times))

plt.tight_layout()
plt.savefig('early_warning_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Step 12: Attention Map Visualization

In [ ]:
# Visualize temporal attention weights for different risk levels
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Temporal Attention Maps by Risk Level\n(High attention = model focuses here for prediction)',
             fontsize=14, fontweight='bold')

t = np.linspace(0, 2, 200)

for risk_level in range(3):
    ax = axes[risk_level]

    # Get a sample
    mask = y == risk_level
    sample = X[mask][0]

    # Normalize
    sample_norm = train_ds.scaler.transform(sample).reshape(1, *sample.shape)
    x_t = torch.FloatTensor(sample_norm).to(DEVICE)

    with torch.no_grad():
        _, attn = model(x_t, return_attention=True)

    attn_np = attn.cpu().numpy()[0]

    # Plot sensor signal (Z-axis accel as reference)
    ax2 = ax.twinx()
    ax2.plot(t, sample[:, 2], color='gray', alpha=0.5, linewidth=1, label='Acc_Z')
    ax2.set_ylabel('Acc_Z (g)', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')

    # Plot attention weights
    color = ['#2ecc71', '#f39c12', '#e74c3c'][risk_level]
    ax.fill_between(t, attn_np, alpha=0.6, color=color)
    ax.plot(t, attn_np, color=color, linewidth=2)
    ax.set_ylabel('Attention Weight', fontweight='bold')
    ax.set_title(f'{RISK_LABELS[risk_level]} - Temporal Attention Map', fontweight='bold')
    ax.grid(True, alpha=0.3)

    if risk_level == 2:
        ax.set_xlabel('Time (seconds)')

    # Highlight top-3 attention peaks
    peaks = np.argsort(attn_np)[-5:]
    for p in peaks:
        ax.axvline(t[p], color='black', alpha=0.4, linestyle=':', linewidth=1)

plt.tight_layout()
plt.savefig('attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Attention maps show where the model focuses in the time series!')

## 🔬 Step 13: Sensor Importance Analysis

In [ ]:
# Sensor ablation study - measure performance drop when each sensor is zeroed out
sensor_groups = {
    'Accelerometer': [0, 1, 2],
    'Gyroscope': [3, 4, 5],
    'EMG': [6, 7],
    'Heart Rate': [8],
    'Force Plate': [9]
}

print('🔬 Sensor Ablation Study (Zeroing out each sensor group)...')
print('='*55)

# Baseline accuracy
_, base_acc, _, _, _ = eval_epoch(model, test_loader, criterion)
print(f'Baseline Accuracy: {base_acc*100:.2f}%')
print('='*55)

ablation_results = {}

for sensor_name, channels in sensor_groups.items():
    # Create ablated test set
    X_test_abl = test_ds.X.clone()
    X_test_abl[:, :, channels] = 0  # Zero out sensor

    abl_ds = torch.utils.data.TensorDataset(X_test_abl, test_ds.y)
    abl_loader = DataLoader(abl_ds, batch_size=BATCH_SIZE, shuffle=False)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in abl_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            preds = model(Xb).argmax(1)
            correct += (preds == yb).sum().item()
            total += len(yb)

    abl_acc = correct / total
    drop = (base_acc - abl_acc) * 100
    ablation_results[sensor_name] = {'acc': abl_acc, 'drop': drop}
    print(f'{sensor_name:15s} → Acc: {abl_acc*100:.2f}% (Drop: {drop:+.2f}%)')

# Visualize ablation study
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Sensor Importance Analysis (Ablation Study)', fontsize=14, fontweight='bold')

sensors = list(ablation_results.keys())
drops = [ablation_results[s]['drop'] for s in sensors]
accs = [ablation_results[s]['acc']*100 for s in sensors]

colors_abl = ['#e74c3c' if d > 3 else '#f39c12' if d > 1 else '#2ecc71' for d in drops]

bars = ax1.barh(sensors, drops, color=colors_abl, edgecolor='black', linewidth=0.8)
for bar, d in zip(bars, drops):
    ax1.text(bar.get_width() + 0.05, bar.get_y()+bar.get_height()/2,
             f'{d:+.2f}%', va='center', fontweight='bold')
ax1.axvline(0, color='black', linewidth=0.5)
ax1.set_xlabel('Accuracy Drop (%)')
ax1.set_title('Performance Drop When Sensor Removed', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

ax2.barh(sensors, accs, color=colors_abl, edgecolor='black', linewidth=0.8)
ax2.axvline(base_acc*100, color='blue', linestyle='--', linewidth=2, label=f'Baseline ({base_acc*100:.1f}%)')
ax2.set_xlabel('Accuracy (%)')
ax2.set_title('Accuracy Without Each Sensor', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('sensor_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Step 14: Final Results Summary

In [ ]:
print('='*65)
print('  🏆 INJURY RISK PREDICTION - FINAL RESULTS SUMMARY')
print('='*65)
print(f'''
📌 Project: Injury Risk Prediction using Multi-Sensor Fusion
📌 Model: CNN-BiLSTM-Attention (InjuryRiskNet)

🔬 SENSOR MODALITIES:
   • Accelerometer (3-axis)  → Impact force detection
   • Gyroscope (3-axis)      → Angular velocity & rotation
   • EMG (2-channel)         → Muscle fatigue patterns
   • Heart Rate Monitor      → Physiological load
   • Force Plate             → Ground reaction forces

🧠 MODEL ARCHITECTURE:
   Input → Channel Attention → Parallel Sensor CNNs
         → Multi-scale CNN → BiLSTM → Temporal Attention
         → MLP Classifier → 3-class Output
   Parameters: {total_params:,}

📊 PERFORMANCE:
   Test Accuracy:   {test_acc*100:.2f}%
   Weighted F1:     {f1:.4f}
   ROC-AUC (OvR):  {auc:.4f}
   Best Val Acc:    {best_val_acc*100:.2f}%

⚠️  EARLY WARNING SYSTEM:
   Warning Threshold:  {ews.warning_threshold:.0%} risk score
   Critical Threshold: {ews.critical_threshold:.0%} risk score
   Lead Time:          {ews.warning_lead_time:.1f} seconds before critical phase
   Window Size:        2.0 seconds (200 samples @ 100Hz)
   Step Size:          0.2 seconds (20 samples)

🔬 MOST IMPORTANT SENSORS (Ablation Study):
''')
sorted_sensors = sorted(ablation_results.items(), key=lambda x: x[1]['drop'], reverse=True)
for rank, (s, v) in enumerate(sorted_sensors, 1):
    print(f'   {rank}. {s:15s}: {v["drop"]:+.2f}% accuracy drop')

print()
print('='*65)
print('✅ Project Complete! All outputs saved.')
print('='*65)

## 💾 Step 15: Save All Outputs

In [ ]:
import zipfile
import os

# Save model and results
results_dict = {
    'test_accuracy': float(test_acc),
    'weighted_f1': float(f1),
    'roc_auc': float(auc),
    'best_val_acc': float(best_val_acc),
    'n_parameters': total_params,
    'sensor_importance': {k: {'accuracy_drop': v['drop']} for k, v in ablation_results.items()},
    'training_epochs': len(history['train_loss'])
}

with open('results_summary.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

# Package everything
output_files = [
    'best_injury_risk_model.pt',
    'sensor_eda.png',
    'training_curves.png',
    'evaluation_metrics.png',
    'early_warning_timeline.png',
    'attention_maps.png',
    'sensor_importance.png',
    'results_summary.json'
]

with zipfile.ZipFile('injury_risk_outputs.zip', 'w') as zf:
    for f in output_files:
        if os.path.exists(f):
            zf.write(f)
            print(f'  ✅ Added: {f}')

print('\n🎉 All outputs packaged in injury_risk_outputs.zip')

# Download in Colab
try:
    from google.colab import files
    files.download('injury_risk_outputs.zip')
    print('📥 Download started!')
except:
    print('💡 In Colab: Run files.download("injury_risk_outputs.zip") to download')